<a href="https://colab.research.google.com/github/Kartiksinghparihar/-DecodeLabs-Internship/blob/main/Project_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import LabelEncoder, StandardScaler
from imblearn.over_sampling import SMOTE
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, roc_auc_score
from imblearn.pipeline import Pipeline
from sklearn.model_selection import RandomizedSearchCV


In [2]:
data = pd.read_csv("/content/fraudTest.csv")


FileNotFoundError: [Errno 2] No such file or directory: '/content/fraudTest.csv'

In [ ]:
data

In [ ]:
data.info()

In [ ]:
data = data.drop([
    'Unnamed: 0',
    'cc_num',
    'first',
    'last',
    'street',
    'trans_num'
], axis=1)

In [ ]:
data['trans_date_trans_time'] = pd.to_datetime(data['trans_date_trans_time'])
data['dob'] = pd.to_datetime(data['dob'])

data["transaction_hour"] = data["trans_date_trans_time"].dt.hour
data["transaction_day"] = data["trans_date_trans_time"].dt.day
data["customer_age"] = (
    data["trans_date_trans_time"].dt.year -
    data["dob"].dt.year
)

In [ ]:
data = data.drop(['trans_date_trans_time', 'dob'], axis=1)


In [ ]:
categorical_columns = [
    'merchant',
    'category',
    'gender',
    'city',
    'state',
    'job'
]

In [ ]:
encoder = LabelEncoder()

In [ ]:
for column in categorical_columns:
    data[column] = encoder.fit_transform(
        data[column].astype(str)
    )

In [ ]:
data = data.dropna()


In [ ]:
X = data.drop("is_fraud", axis=1)
y = data["is_fraud"]

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [ ]:
logistic_pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("smote", SMOTE(random_state=42)),
    ("model", LogisticRegression(max_iter=1000))
])

In [ ]:
logistic_pipeline.fit(X_train, y_train)


In [ ]:
lr_predictions = logistic_pipeline.predict(X_test)
lr_probabilities = logistic_pipeline.predict_proba(X_test)[:, 1]

In [ ]:
print(classification_report(y_test, lr_predictions))

In [ ]:
lr_auc = roc_auc_score(
    y_test,
    lr_probabilities
)

In [ ]:
print("ROC-AUC Score:", lr_auc)

In [ ]:
rf_pipeline = Pipeline([
    ("smote", SMOTE(random_state=42)),
    ("model", RandomForestClassifier(random_state=42))
])

In [ ]:
parameters = {
    "model__n_estimators": [100, 200],
    "model__max_depth": [10, 20, None]
}

In [ ]:
grid_search = GridSearchCV(
    rf_pipeline,
    param_grid=parameters,
    cv=3,
    scoring="recall",
    n_jobs=-1
)


In [ ]:
random_search = RandomizedSearchCV(
    rf_pipeline,
    param_distributions={
        "model__n_estimators": [50, 100, 150],
        "model__max_depth": [5, 10, 15, None]
    },
    n_iter=4,
    cv=2,
    scoring="recall",
    random_state=42,
    n_jobs=-1
)

random_search.fit(X_train, y_train)